In [ ]:
import numpy as np
import pyvista as pv
from scipy.spatial import Voronoi, ConvexHull

def generate_3d_clipped_voronoi(seeds, bounds=[0, 300, 0, 300, 0, 300]):
    """Generates 3D Voronoi cells and clips them to a 3D box."""
    vor = Voronoi(seeds)
    xmin, xmax, ymin, ymax, zmin, zmax = bounds
    
    # Define the 8 corners of the 3D RVE box for clipping
    rve_box = pv.Box(bounds)
    
    cells = []
    for region_idx in vor.point_region:
        region = vor.regions[region_idx]
        if not region or -1 in region: continue
        
        # 1. Create the 3D convex hull for the Voronoi cell
        verts = vor.vertices[region]
        hull = ConvexHull(verts)
        
        # 2. Convert to PyVista PolyData
        faces = np.column_stack((np.full(len(hull.simplices), 3), hull.simplices)).flatten()
        cell_mesh = pv.PolyData(verts, faces)
        
        # 3. Clip the cell mesh with the RVE bounding box
        # This ensures perfectly flat faces on the X, Y, and Z planes
        clipped_cell = cell_mesh.clip_box(bounds, invert=False)
        
        if clipped_cell.n_points > 0:
            cells.append(clipped_cell)
            
    return cells

def visualize_3d_rve(clipped_cells):
    plotter = pv.Plotter()
    
    for i, cell in enumerate(clipped_cells):
        # Apply a random color to each 3D grain
        color = np.random.random(3)
        plotter.add_mesh(cell, color=color, edge_color='k', 
                         show_edges=True, opacity=0.8)
    
    plotter.show_grid()
    plotter.add_axes()
    plotter.show()

In [ ]:
import numpy as np
import pyvista as pv
from scipy.spatial import Voronoi, ConvexHull

def generate_3d_test_setup(n_seeds=20, size=100.0, padding=20.0):
    # 1. Generate 20 random seeds within the 100x100x100 box
    inner_seeds = np.random.uniform(0, size, (n_seeds, 3))
    
    # 2. Add Guard Plane seeds to ensure flat RVE faces
    # We place seeds outside each face to force perpendicular bisectors
    guards = []
    offsets = [-padding, size + padding]
    for i in range(3): # X, Y, Z axes
        for val in offsets:
            # Create a grid of points on the guard plane
            grid = np.linspace(0, size, 5)
            m1, m2 = np.meshgrid(grid, grid)
            plane = np.zeros((25, 3))
            plane[:, i] = val
            plane[:, (i+1)%3] = m1.flatten()
            plane[:, (i+2)%3] = m2.flatten()
            guards.append(plane)
    
    all_seeds = np.vstack([inner_seeds] + guards)
    return all_seeds, inner_seeds

def test_3d_voronoi_visualization(all_seeds, inner_n, size=100.0):
    vor = Voronoi(all_seeds)
    bounds = [0, size, 0, size, 0, size]
    plotter = pv.Plotter()
    
    # Only iterate through the original 20 points (ignore guards)
    for i in range(inner_n):
        region_idx = vor.point_region[i]
        region = vor.regions[region_idx]
        
        if -1 in region or not region: continue
        
        # Build the solid polyhedron
        verts = vor.vertices[region]
        hull = ConvexHull(verts)
        faces = np.column_stack((np.full(len(hull.simplices), 3), hull.simplices)).flatten()
        
        cell_mesh = pv.PolyData(verts, faces)
        
        # Clip to the RVE box
        clipped_cell = cell_mesh.clip_box(bounds, invert=False)
        
        # Visualize with 3D edge lines to inspect the 'welding' points
        plotter.add_mesh(clipped_cell, 
                         color=np.random.random(3), 
                         opacity=0.7, 
                         show_edges=True, 
                         edge_color='white',
                         line_width=2)

    plotter.show_grid()
    plotter.add_axes()
    plotter.show()

# Run the test
all_pts, inner_pts = generate_3d_test_setup(n_seeds=50)
test_3d_voronoi_visualization(all_pts, len(inner_pts))

In [ ]:
import numpy as np
import pyvista as pv
from scipy.spatial import Voronoi, ConvexHull

def generate_3d_rve_base(n_seeds=20, size=100.0, padding=25.0):
    """
    Step 1: Generate 3D seeds with Guard Planes to enforce 
    perpendicularity at the RVE boundaries.
    """
    # Stochastic inner seeds
    inner_seeds = np.random.uniform(0, size, (n_seeds, 3))
    
    # Generate Guard Planes (6 planes total)
    guards = []
    plane_res = 5 # Number of seeds per guard plane
    grid = np.linspace(0, size, plane_res)
    m1, m2 = np.meshgrid(grid, grid)
    
    offsets = [-padding, size + padding]
    for i in range(3): # X, Y, Z
        for val in offsets:
            plane = np.zeros((plane_res**2, 3))
            plane[:, i] = val
            plane[:, (i+1)%3] = m1.flatten()
            plane[:, (i+2)%3] = m2.flatten()
            guards.append(plane)
            
    all_seeds = np.vstack([inner_seeds] + guards)
    return all_seeds, len(inner_seeds)

def render_3d_test(all_seeds, inner_count, size=100.0):
    """
    Step 2: Compute Voronoi, clip cells, and render.
    """
    vor = Voronoi(all_seeds)
    bounds = [0, size, 0, size, 0, size]
    plotter = pv.Plotter()
    
    # We only care about cells associated with the original 20 points
    for i in range(inner_count):
        region_idx = vor.point_region[i]
        region = vor.regions[region_idx]
        
        if -1 in region or not region: continue
        
        # Build 3D Polyhedron from vertices
        verts = vor.vertices[region]
        hull = ConvexHull(verts)
        # PyVista expects: [n_points, i, j, k, ...]
        faces = np.column_stack((np.full(len(hull.simplices), 3), hull.simplices)).flatten()
        
        cell_mesh = pv.PolyData(verts, faces)
        
        # Clip the cell strictly to the RVE box
        # This is where we verify if the Guard Planes worked
        clipped_cell = cell_mesh.clip_box(bounds, invert=False)
        
        # Plot each cell with unique color and visible edges
        plotter.add_mesh(
            clipped_cell, 
            color=np.random.random(3), 
            opacity=0.9, 
            show_edges=True, 
            edge_color='black',
            line_width=1
        )

    plotter.show_grid()
    plotter.add_axes()
    plotter.view_isometric()
    plotter.show()

# Execution
seeds_total, count = generate_3d_rve_base(n_seeds=200)
render_3d_test(seeds_total, count)

In [ ]:
import numpy as np
import pyvista as pv
from scipy.spatial import Voronoi, ConvexHull

def generate_iron_box_seeds(n_inner=1000, size=100.0, guard_dist=50.0):
    # 1. Stochastic inner seeds
    inner_seeds = np.random.uniform(0, size, (n_inner, 3))
    
    # 2. Dense Guard Sheets (6 total)
    # Increase guard_res if edges still look "bumpy"
    guard_res = 10 
    grid = np.linspace(0, size, guard_res)
    m1, m2 = np.meshgrid(grid, grid)
    
    guards = []
    # Project sheets for X=0, X=size, Y=0, Y=size, Z=0, Z=size
    for i in range(3): 
        for val in [-guard_dist, size + guard_dist]:
            sheet = np.zeros((guard_res**2, 3))
            sheet[:, i] = val
            sheet[:, (i+1)%3] = m1.flatten()
            sheet[:, (i+2)%3] = m2.flatten()
            guards.append(sheet)
            
    all_seeds = np.vstack([inner_seeds] + guards)
    return all_seeds, n_inner

def render_solid_rve(all_seeds, n_inner, size=100.0):
    vor = Voronoi(all_seeds)
    bounds = [0, size, 0, size, 0, size]
    plotter = pv.Plotter()
    
    for i in range(n_inner):
        region = vor.regions[vor.point_region[i]]
        if -1 in region or not region: continue
        
        # Build polyhedron
        verts = vor.vertices[region]
        hull = ConvexHull(verts)
        faces = np.column_stack((np.full(len(hull.simplices), 3), hull.simplices)).flatten()
        
        # The key verification step
        mesh = pv.PolyData(verts, faces)
        # clip_box ensures the final visual is a perfect cube
        cleaned_mesh = mesh.clip_box(bounds, invert=False)
        plotter.add_mesh(cleaned_mesh, color=np.random.random(3), style='wireframe',
                         show_edges=True, 
                         edge_color='gray', line_width=1, opacity=1.0, edge_opacity=0.25,
                         show_vertices=True, point_size=2
                         )

    #plotter.show_grid()
    plotter.show()

# Test the "Iron Box"
seeds, count = generate_iron_box_seeds(250)
render_solid_rve(seeds, count)